# 1. Disponibilizar el modelo


In [289]:
import pandas as pd
import numpy as np
import joblib

In [290]:
# Carga de datos de archivo .csv
dataTraining = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2025/main/datasets/dataTrain_Spotify.csv')
dataTesting = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2025/main/datasets/dataTest_Spotify.csv', index_col=0)

In [291]:
dataTraining = dataTraining[['artists', 'track_genre', 'duration_ms', 'popularity', 'explicit', 'danceability']]

In [292]:
dataTraining[['artists', 'popularity']].sort_values(by='popularity', ascending= False).head(10)

,artists,popularity
50942,Sam Smith;Kim Petras,100
74999,Sam Smith;Kim Petras,100
31491,Bizarrap;Quevedo,99
17924,Manuel Turizo,98
71798,Manuel Turizo,98
71976,David Guetta;Bebe Rexha,98
50494,David Guetta;Bebe Rexha,98
38479,Manuel Turizo,98
60016,Bad Bunny;Chencho Corleone,97
69360,Bad Bunny,97


In [293]:
dataTraining[['track_genre', 'popularity']].sort_values(by='popularity', ascending= False).head()

,track_genre,popularity
50942,dance,100
74999,pop,100
31491,hip-hop,99
17924,reggae,98
71798,latin,98


In [294]:
def cleandf(df):
  try:
    df.drop(columns=['Unnamed: 0','track_id' , 'track_name'], inplace=True)
  except KeyError: # Handle specific KeyError if columns are not found
    pass
  # Return the modified DataFrame
  return df

# Label encoder
def labelencoder(df):
  from sklearn.preprocessing import LabelEncoder
  categorical_vars = ['artists', 'track_genre']
  le = LabelEncoder()
  # Apply fit_transform to each column separately
  for col in categorical_vars:
    if col in df.columns:  # Check if the column exists
      df[col] = le.fit_transform(df[col])
  # Return the modified DataFrame
  return df

dataTraining = cleandf(dataTraining)
dataTraining = labelencoder(dataTraining)

In [295]:
#Agrupación
def group(df):
  try:
    group_training = dataTraining.groupby(['artists', 'track_genre', 'explicit'])[['duration_ms', 'danceability', 'popularity' ]].median().reset_index()
  except:
    group_training = dataTraining.groupby(['artists', 'album_name', 'track_genre', 'explicit', 'key','mode', 'time_signature'])[['duration_ms', 'danceability','energy', 'loudness','speechiness', 'acousticness','instrumentalness', 'liveness','valence','tempo']].median().reset_index()
  return group_training

group_training = group(dataTraining)

In [296]:
#sqrt
def sqrt(df):
  numerical_vars = [ 'duration_ms', 'danceability']
  try:
    for i in numerical_vars:
      df[i] = np.sqrt(dataTraining[i])
  except:
    pass
  return df

group_training = sqrt(group_training)

In [298]:
from sklearn.model_selection import train_test_split

x = group_training.drop(columns=['popularity'])
y = group_training['popularity']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=123)

In [299]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error

xg = xgb.XGBRegressor(n_estimators=125, random_state=123, gamma =0, learning_rate =0.1, max_depth =18, colsample_bytree = 0.8)
xg.fit(x_train, y_train)

y_pred = xg.predict(x_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE:", rmse)

RMSE: 18.481990629886837


In [301]:
joblib.dump(xg, 'randomforestMl', compress = 3)

['randomforestMl']

In [302]:
!pip install flask-restx

In [303]:
from flask import Flask, request, jsonify
from flask_restx import Api, Resource, fields

app = Flask(__name__)

# Definición API Flask
api = Api(
    app, 
    version='1.0', 
    title='PrediccionMl',
    description='PrediccionMl')

ns = api.namespace('PrediccionMlXG', 
     description='PrediccionMlXG')

# Definición argumentos o parámetros de la API
parser = ns.parser()
parser.add_argument(
  'artists',
  type=str,
  required=True,
  help='Ingresa el nombre del artista',
  location='args')

parser.add_argument(
  'track_genre',
  type=str,
  required=True,
  help='Ingresa el genero de la canción',
  location='args')

parser.add_argument(
  'duration_ms',
  type=float,
  required=True,
  help='Duraciòn en milisegundos entre 0 y 200000',
  location='args')

parser.add_argument(
  'explicit',
  type=int,
  required=True,
  help='Explicit (1 for yes, 0 for no)',
  location='args')

parser.add_argument(
  'danceability',
  type=float,
  required=True,
  help='Ingrese el danceability entre 0 y 1',
  location='args')



resource_fields = api.model('Resource', {
    'result': fields.String,
})

In [305]:
# Disponibilizaciòn del modelo api
# Disponibilizaciòn del modelo api

@ns.route('/')
class PrediccionMlXG(Resource):

  @api.doc(parser=parser)
  @api.marshal_with(resource_fields)
  def get(self):
      args = parser.parse_args()

      # Crear un DataFrame con los datos de entrada
      input_data = pd.DataFrame([args])
      
      # Asegurarse de que las columnas estén en el mismo orden que durante el entrenamiento

      input_data = input_data[['artists', 'track_genre',  'explicit', 'duration_ms', 'danceability']]

  
      #Label encoder
      input_data = labelencoder(input_data)

      #sqrt
      input_data = sqrt(input_data)

      # Realizar la predicción
      prediction = xg.predict(input_data)[0]  # Tomar el primer elemento de la predicción

      return {
       "result": f'La popularidad de tu canción es: {prediction}'
      }, 200

if __name__ == '__main__':
  app.run(debug=True, use_reloader=False, host='0.0.0.0', port=5000)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.40:5000
Press CTRL+C to quit
